In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.stats import sigma_clipped_stats

from sklearn.cluster import DBSCAN

from Kakapo.photometry import forced_photometry

%matplotlib widget

In [2]:
def initial_filter(df):
    df = df[(df.fwhm > 0.8) & (df.snr >= 1) & 
            (df.psfdiff <= 2) & (df.poisson_thresh >= 0.5) & 
            (abs(df.correlation) >= 0.05)]
    
    return df

def _grouping(corr: pd.DataFrame, f_dist: int = 48) -> pd.DataFrame | None:
    """
    Group detections with DBSCAN in O(N log N) time, using only C/Fortran
    code paths from scikit-learn (no Python callback per point pair).
    """
    if corr.empty:
        return None

    # Scale the frame axis so that `eps` of 1.25 encloses ±f_dist frames.
    data = corr[['xcentroid', 'ycentroid', 'frame']].values.astype(np.float32)
    data[:, 2] *= 1.5 / f_dist

    db = DBSCAN(eps=1.5,
                min_samples=5,
                metric='euclidean',           # now fully compiled
                algorithm='auto',        # fastest for 3‑D Euclidean
                n_jobs=1)                     # keep it serial – you already parallelise at a higher level
    labels = db.fit_predict(data)

    corr = corr.assign(cluster=labels)
    corr = corr[corr.cluster != -1]           # drop noise points

    return corr if not corr.empty else None

# def mask_detections(correlation, psfdiff, fwhm, snr, 
#                     roundness, poisson_thresh, xstd, ystd):
    
#     mask =  (correlation >= self.corrlim) & (psfdiff <= self.difflim) & \
#             (fwhm <= self.fwhmlim) & (fwhm >= 0.9) & \
#             (snr >= self.snrlim) & (snr < 10000) & (abs(roundness) <= self.roundness) & \
#             (poisson_thresh >= self.poiss_val) & \
#             (xstd <= self.dist_cut) & (ystd <= self.dist_cut)
    
    
#     return mask

In [3]:
file = '/Users/zgl12/Modules/Kakapo/Data/csv_files/c3/c3_t205922648.csv'
df = pd.read_csv(file)

In [4]:
df_new = df[(df['fwhm'] >= 0.8) & (df['fwhm'] <= 5) & 
   (df['roundness'] <= 0.95) & (abs(df['correlation'])>= 0.2) & 
   (abs(df['snr'])>= 3) & (abs(df['snr'])<= 1e4) & (df['psfdiff'] <= 1.2) & 
   (df['psfdiff'] <= 1.2) & (df['poisson_thresh'] >= 1)]

In [5]:
# plt.figure()
# plt.scatter(df.xcentroid.values, df.ycentroid.values, c = df.frame.values)
# plt.xlabel(r'$x$')
# plt.ylabel(r'$y$')
# plt.show()

# plt.figure()
# plt.scatter(df.fwhm.values, df.roundness.values, c = df.frame.values)
# plt.xlabel(r'FWHM')
# plt.ylabel(r'Roundness')
# plt.show()

# plt.figure()
# plt.scatter(df.snr.values, df.correlation.values, c = df.frame.values)
# plt.xlabel(r'SNR')
# plt.ylabel(r'Correlation')
# plt.show()

# plt.figure()
# plt.scatter(df.psfdiff.values, df.poisson_thresh.values, c = df.frame.values)
# plt.xlabel(r'PSF-Diff.')
# plt.ylabel(r'Poisson Threshold')
# plt.show()

In [6]:
df_1 = initial_filter(df)

df_1 = _grouping(df_1, f_dist = 48)

In [7]:
df_1

,xcentroid,ycentroid,fwhm,roundness,pa,max_value,flux,mag,snr,flux_err,...,n_detections,poisson_thresh,ref_flux,campaign,target_id,ra,dec,filename,ref_frame,cluster
44,8.452718,6.000000,0.828824,1.000000,90.000000,1.913508,3.496385,-1.359048,1.0,3.496385,...,1,0.840364,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
45,8.371919,6.000000,0.804775,1.000000,90.000000,1.213115,1.931461,-0.714715,1.0,1.931461,...,1,0.532789,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
52,8.000000,6.597964,0.816418,1.000000,0.000000,2.105737,3.521511,-1.366823,1.0,3.521511,...,1,0.918016,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
53,8.000000,6.616360,0.809696,1.000000,0.000000,2.180344,3.537454,-1.371727,1.0,3.537454,...,1,0.950999,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
61,8.488483,7.000000,0.832334,1.000000,90.000000,2.501410,4.890184,-1.723313,1.0,4.890184,...,1,1.084233,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
71,8.381223,6.000000,0.808722,1.000000,90.000000,1.920618,3.103895,-1.229767,1.0,3.103895,...,1,0.843209,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
72,8.397040,6.000000,0.814712,1.000000,90.000000,2.269488,0.929845,0.078974,1.0,0.929845,...,1,0.996798,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,0
84,7.000000,7.394854,0.813938,1.000000,0.000000,2.248759,3.716059,-1.425207,1.0,3.716059,...,1,0.971128,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,1
88,8.000000,6.561198,0.826295,1.000000,0.000000,3.227646,0.905451,0.107838,1.0,0.905451,...,1,1.406637,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,1
97,7.000000,7.411871,0.819520,1.000000,0.000000,3.474724,5.908094,-1.928619,1.0,5.908094,...,1,1.500132,1636.899423,3,205922648,337.397715,-17.325794,./Data/csv_files/c3/c3_t205922648.csv,Median,1


In [ ]:
for cluster in np.unique(df_1.cluster.values):
    
    temp_df = df_1[df_1.cluster == cluster]
    print(cluster)
    print(len(temp_df), temp_df.frame.min(), temp_df.frame.max())
    x, _, xstd = sigma_clipped_stats(temp_df.xcentroid.values, sigma = 3)
    y, _, ystd = sigma_clipped_stats(temp_df.ycentroid.values, sigma = 3)
    
    print(f"{x:.2f} +/- {xstd:.2f}")
    print(f"{y:.2f} +/- {ystd:.2f}")
    
    # x, _, xstd = sigma_clipped_stats(temp_df.xcentroid.values, sigma = 3)
    # y, _, ystd = sigma_clipped_stats(temp_df.ycentroid.values, sigma = 3)
    correlation, _, _ = sigma_clipped_stats(temp_df.correlation.values, sigma = 3)
    psfdiff, _, _  = sigma_clipped_stats(temp_df.psfdiff.values, sigma = 3)
    snr, _, _  = sigma_clipped_stats(temp_df.snr.values, sigma = 3)
    fwhm, _, _  = sigma_clipped_stats(temp_df.fwhm.values, sigma = 3)
    roundness, _, _  = sigma_clipped_stats(temp_df.roundness.values, sigma = 3)
    poisson_thresh, _, _  = sigma_clipped_stats(temp_df.poisson_thresh.values, sigma = 3)

    print('Corr.', correlation)
    print('PSF Diff.', psfdiff)
    print('SNR', snr)
    print('FWHM', fwhm)
    print('Roundness', roundness)
    print('Poiss.', poisson_thresh)
    print()


In [ ]:


# mask_detections(correlation, psfdiff, fwhm, snr, 
#                     roundness, poisson_thresh, xstd, ystd)

In [ ]:
diff = np.load('/Users/zgl12/Modules/Kakapo/Data/difference_arrays/c3/diff_c3_t205922648.npy')

In [ ]:
fluxes = forced_photometry(diff, 6.36, 6.35, None)

In [ ]:
plt.figure()
plt.plot(fluxes)
plt.axvline(2695.0, color = 'r')
plt.axvline(3385.0, color = 'r')
plt.show()